In [0]:
!pip install -qU torch
!pip install -q transformers==4.44.1
!pip install -q accelerate
!pip install -q bitsandbytes==0.43.3
dbutils.library.restartPython()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.18.1+cu121 requires torch==2.3.1, but you have torch 2.5.1 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

2025-01-21 10:33:10.819664: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-21 10:33:10.854139: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [0]:
def print_gpu_memory():
    if torch.cuda.is_available():
        print(
            "{:<10} {:<15} {:<15} {:<15}".format(
                "GPU", "Total (GB)", "Allocated (GB)", "Available (GB)"
            )
        )
        print("-" * 60)
        for i in range(torch.cuda.device_count()):
            total_memory = torch.cuda.get_device_properties(i).total_memory / 1e9
            allocated_memory = torch.cuda.memory_allocated(i) / 1e9
            available_memory = total_memory - allocated_memory
            print(
                "{:<10} {:<15} {:<15} {:<15}".format(
                    f"GPU_{i}",
                    f"{total_memory:.2f}",
                    f"{allocated_memory:.2f}",
                    f"{available_memory:.2f}",
                )
            )
            print()
    else:
        print("No GPU available.")

In [0]:
print_gpu_memory()

GPU        Total (GB)      Allocated (GB)  Available (GB) 
------------------------------------------------------------
GPU_0      99.88           0.00            99.88          



In [0]:
hf_token = "<your-token-here>"

In [0]:
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name, device_map = "auto", token=hf_token)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [0]:
print_gpu_memory()

GPU        Total (GB)      Allocated (GB)  Available (GB) 
------------------------------------------------------------
GPU_0      99.88           32.12           67.75          



In [0]:
param_dtypes = [param.dtype for param in model.parameters()]
print("Parameter dtypes:", param_dtypes[:10])

print(model.get_memory_footprint()) # bytes
print(model.get_memory_footprint()/(1024*1024*1024)) # GB

Parameter dtypes: [torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32, torch.float32]
32121053440
29.915062189102173


In [0]:
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
input = tokenizer("Portugal is", return_tensors="pt").to('cuda')

response = model.generate(**input, max_new_tokens = 50)
print(tokenizer.batch_decode(response, skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


["Portugal is one of the most popular tourist destinations in the world, and for good reason. From the beautiful beaches to the rich history and culture, there's something for everyone in Portugal. Here are some of the top things to do and see in Portugal:\n1"]


### Clear GPU

In [0]:
# del model  # Delete the model object
# torch.cuda.empty_cache()  # Clear the GPU cache
# torch.cuda.synchronize()  # Ensure all operations on GPU are completed

# print("GPU memory has been cleared.")
# print_gpu_memory()

### Load 8-bit quantized model

In [0]:
bnb_config = BitsAndBytesConfig(
    load_in_8bit = True
)

In [0]:
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
quantized_model = AutoModelForCausalLM.from_pretrained(model_name,
                    quantization_config = bnb_config,
                    token=hf_token,
                    device_map = "auto")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [0]:
param_dtypes = [param.dtype for param in quantized_model.parameters()]
print("Parameter dtypes:", param_dtypes[:10])

print(quantized_model.get_memory_footprint()) # bytes
print(quantized_model.get_memory_footprint()/(1024*1024*1024)) # GB

Parameter dtypes: [torch.float16, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.int8, torch.float16, torch.float16]
9081209088
8.457535028457642


In [0]:
# tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
# input = tokenizer("Portugal is", return_tensors="pt").to('cuda')

# response = quantized_model.generate(**input, max_new_tokens = 50)
# print(tokenizer.batch_decode(response, skip_special_tokens=True))

### Load 4-bit quantized model

In [0]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True
)

In [0]:
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
quantized_model = AutoModelForCausalLM.from_pretrained(model_name,
                    quantization_config = bnb_config,
                    token=hf_token,
                    # load_in_8bit=True, # deprecated
                    # load_in_4bit=True, # deprecated
                    device_map = "auto")

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [0]:
param_dtypes = [param.dtype for param in quantized_model.parameters()]
print("Parameter dtypes:", param_dtypes[:10])

print(quantized_model.get_memory_footprint()) # bytes
print(quantized_model.get_memory_footprint()/(1024*1024*1024)) # GB

Parameter dtypes: [torch.float16, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.float16, torch.float16]
5591548160
5.207535028457642


In [0]:
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
input = tokenizer("Portugal is", return_tensors="pt").to('cuda')

response = quantized_model.generate(**input, max_new_tokens = 50)
print(tokenizer.batch_decode(response, skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-349b1880-bc1d-4ebd-8f2b-e24e3c90bc15/lib/python3.11/site-packages/bitsandbytes/nn/modules.py:435: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(


['Portugal is a beautiful country with a rich history and a vibrant culture. The country is known for its stunning beaches, picturesque villages, and historic cities, including Lisbon, Porto, and the Algarve region. Visitors can explore the historic neighborhoods of Lisbon, visit']
